In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import from_json, col, udf, when, broadcast
from pyspark.sql.types import *
import joblib
import numpy as np
import pandas as pd
import requests
import psycopg2
from datetime import datetime

#---------------------------------------------------------------------------------
#connect to timescaledb
DB_CONFIG = {
    "host": "localhost",
    "port": 5433,
    "database": "flights_db",
    "user": "admin",
    "password": "admin"
}

#--------------------------------------------------------------------------
# كتابه اسماء ال topics في كافكا 
KAFKA_BROKER = "localhost:9092"
TOPICS = {
    "flights": "flight_raw",
    "weather": "weather_raw"
}

#------------------------------------------
# تحميل المودل من اللوكل 
MODEL_PATH = "/home/ahmed-refat/Desktop/flights & airports/ML model"
model = joblib.load(f"{MODEL_PATH}/xgboost_delay_model.pkl")
feature_names = joblib.load(f"{MODEL_PATH}/feature_names.pkl")

print(f"Model loaded!")
print(f"Features: {feature_names}")

#-----------------------------------------------------------
# create sparksession 
spark = SparkSession.builder \
    .appName("FlightDelayStream") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.13:4.1.0") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("Spark ready!")

#----------------------------------------------------------------
# write schema for flight stream 
flight_schema = StructType([
    StructField("icao24", StringType()),
    StructField("callsign", StringType()),
    StructField("origin_country", StringType()),
    StructField("longitude", DoubleType()),
    StructField("latitude", DoubleType()),
    StructField("altitude", DoubleType()),
    StructField("velocity", DoubleType()),
    StructField("heading", DoubleType()),
    StructField("on_ground", BooleanType()),
    StructField("timestamp", LongType())
])

#----------------------------------------------
# airline mapping من callsign
airline_mapping = {
    "DAL": "DL", "AAL": "AA", "UAL": "UA",
    "SWA": "WN", "JBU": "B6", "NKS": "NK",
    "RPA": "YX", "EDV": "9E", "GJS": "G7",
    "LXJ": "XO", "EJA": "XO", "BAW": "BA",
    "AIC": "AI"
}

airline_codes = list(set(airline_mapping.values()))
airline_to_int = {a: i for i, a in enumerate(airline_codes)}

def predict_delay(callsign, lat, lon,
                  temperature, wind_speed, wind_gust,
                  precipitation, visibility, humidity, cloudcover):
    try:
        # استخرج airline code من callsign
        prefix = callsign[:3] if callsign else "UNK"
        airline = airline_mapping.get(prefix, "UNK")
        airline_int = airline_to_int.get(airline, -1)

        features = pd.DataFrame([[
            airline_int, lat, lon,
            temperature, wind_speed, wind_gust,
            precipitation, visibility, humidity, cloudcover
        ]], columns=feature_names)

        pred = model.predict(features)[0]
        return int(pred)
    except:
        return -1

predict_udf = udf(predict_delay, IntegerType())

#------------------------------------------
# جيب الـ weather مرة واحدة كـ static من API مباشرة
# عشان نتجنب مشكلة stream-stream join
def get_weather_static():
    NY_AIRPORTS = {
        "JFK": {"lat": 40.6413, "lon": -73.7781},
        "LGA": {"lat": 40.7769, "lon": -73.8740},
        "EWR": {"lat": 40.6895, "lon": -74.1745}
    }
    records = []
    for airport, coords in NY_AIRPORTS.items():
        url = "https://api.open-meteo.com/v1/forecast"
        params = {
            "latitude": coords["lat"],
            "longitude": coords["lon"],
            "current": "temperature_2m,wind_speed_10m,wind_gusts_10m,precipitation,visibility,relative_humidity_2m,cloud_cover",
            "wind_speed_unit": "ms"
        }
        res = requests.get(url, params=params).json()
        current = res["current"]
        records.append({
            "airport": airport,
            "weather_lat": coords["lat"],
            "weather_lon": coords["lon"],
            "temperature": float(current["temperature_2m"]),
            "wind_speed": float(current["wind_speed_10m"]),
            "wind_gust": float(current["wind_gusts_10m"]),
            "precipitation": float(current["precipitation"]),
            "visibility": float(current.get("visibility", 10000.0)),
            "humidity": float(current["relative_humidity_2m"]),
            "cloudcover": float(current["cloud_cover"])
        })
    return records

# تحويل الـ weather لـ Spark static DataFrame
weather_records = get_weather_static()
weather_static = spark.createDataFrame(pd.DataFrame(weather_records))
print("Weather static loaded!")
weather_static.show()

#-------------------------------------------------------------
# القراءه من كافكا - flights فقط
# الـ weather بقا static مش stream
flights_df = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", KAFKA_BROKER) \
    .option("subscribe", TOPICS["flights"]) \
    .option("startingOffsets", "latest") \
    .load() \
    .select(from_json(col("value").cast("string"), flight_schema).alias("data")) \
    .select("data.*")

print("Stream ready!")

#------------------------------
# rename عشان منتلخبطش بعد الجوين
flights_renamed = flights_df.select(
    col("callsign"),
    col("origin_country"),
    col("latitude").alias("flight_lat"),
    col("longitude").alias("flight_lon"),
    col("altitude"),
    col("velocity"),
    col("on_ground"),
    col("timestamp").alias("flight_timestamp")
)

#-------------------------------------------------------------
# join بين flights stream و weather static
# broadcast عشان الـ weather صغير (3 rows بس)
joined_df = flights_renamed.join(
    broadcast(weather_static),
    (flights_renamed.flight_lon.between(
        weather_static.weather_lon - 0.5,
        weather_static.weather_lon + 0.5
    )) &
    (flights_renamed.flight_lat.between(
        weather_static.weather_lat - 0.5,
        weather_static.weather_lat + 0.5
    )),
    "left"
)

#-------------------------------------------------------------
# تطبيق المودل وإضافة عمود is_delayed
result_df = joined_df.withColumn(
    "is_delayed",
    predict_udf(
        col("callsign"),
        col("flight_lat"),
        col("flight_lon"),
        col("temperature"),
        col("wind_speed"),
        col("wind_gust"),
        col("precipitation"),
        col("visibility"),
        col("humidity"),
        col("cloudcover")
    )
).select(
    col("callsign"),
    col("origin_country"),
    col("flight_lat"),
    col("flight_lon"),
    col("on_ground"),
    col("temperature"),
    col("wind_speed"),
    col("is_delayed"),
    when(col("is_delayed") == 1, "DELAYED")
    .when(col("is_delayed") == 0, "ON TIME")
    .otherwise("UNKNOWN").alias("status")
)

#------------------------------------------------------------
# write to timescaledb 
def write_to_timescale(df, epoch_id):
    rows = df.collect()
    if not rows:
        return
    
    conn = psycopg2.connect(**DB_CONFIG)
    cursor = conn.cursor()
    
    for row in rows:
        cursor.execute("""
            INSERT INTO flight_predictions 
            (time, callsign, origin_country, flight_lat, flight_lon, 
             on_ground, temperature, wind_speed, is_delayed, status)
            VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
        """, (
            datetime.now(),
            row["callsign"],
            row["origin_country"],
            row["flight_lat"],
            row["flight_lon"],
            row["on_ground"],
            row["temperature"],
            row["wind_speed"],
            row["is_delayed"],
            row["status"]
        ))
    
    conn.commit()
    cursor.close()
    conn.close()
    print(f"Batch {epoch_id}: {len(rows)} rows written to TimescaleDB")

# writeStream لـ TimescaleDB
query = result_df.writeStream \
    .outputMode("append") \
    .foreachBatch(write_to_timescale) \
    .option("checkpointLocation", "/tmp/checkpoint/timescale") \
    .queryName("timescale_stream") \
    .start()

print("Writing to TimescaleDB...")
spark.streams.awaitAnyTermination()

Model loaded!
Features: ['airline', 'airport_lat', 'airport_lon', 'temperature', 'wind_speed', 'wind_gust', 'precipitation', 'visibility', 'humidity', 'cloudcover']


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/25 07:40:00 WARN Utils: Your hostname, refat, resolves to a loopback address: 127.0.1.1; using 10.0.2.15 instead (on interface enp0s3)
26/04/25 07:40:00 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/home/ahmed-refat/miniconda3/envs/spark/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/ahmed-refat/.ivy2.5.2/cache
The jars for the packages stored in: /home/ahmed-refat/.ivy2.5.2/jars
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-d9100a7d-c46e-415d-9412-c861ecd3d408;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.13;4.1.0 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.13;4.1.0 in central
	found org.apache.kafka#kafka-clients;3.9.1 i

Spark ready!
Weather static loaded!


+-------+-----------+-----------+-----------+----------+---------+-------------+----------+--------+----------+
|airport|weather_lat|weather_lon|temperature|wind_speed|wind_gust|precipitation|visibility|humidity|cloudcover|
+-------+-----------+-----------+-----------+----------+---------+-------------+----------+--------+----------+
|    JFK|    40.6413|   -73.7781|        7.2|      1.87|      6.3|          0.0|   13800.0|    90.0|      24.0|
|    LGA|    40.7769|    -73.874|        7.9|      1.43|      6.2|          0.0|   18700.0|    78.0|       6.0|
|    EWR|    40.6895|   -74.1745|        9.8|      1.66|      7.1|          0.0|   22800.0|    71.0|      18.0|
+-------+-----------+-----------+-----------+----------+---------+-------------+----------+--------+----------+

Stream ready!
Writing to TimescaleDB...


Batch 1: 66 rows written to TimescaleDB


...

26/04/26 00:46:58 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:132)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$driverEndpoint(BlockManagerMasterEndpoint.scala:131)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.isExecutorAlive$lzycompute$1(BlockManagerMasterEndpoint.scala:707)
	at org.apache.spark.storage.BlockManagerMasterE

Batch 90: 71 rows written to TimescaleDB


Batch 91: 206 rows written to TimescaleDB


----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 35936)
Traceback (most recent call last):
  File "/home/ahmed-refat/miniconda3/envs/spark/lib/python3.10/socketserver.py", line 316, in _handle_request_noblock
    self.process_request(request, client_address)
  File "/home/ahmed-refat/miniconda3/envs/spark/lib/python3.10/socketserver.py", line 347, in process_request
    self.finish_request(request, client_address)
  File "/home/ahmed-refat/miniconda3/envs/spark/lib/python3.10/socketserver.py", line 360, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "/home/ahmed-refat/miniconda3/envs/spark/lib/python3.10/socketserver.py", line 747, in __init__
    self.handle()
  File "/home/ahmed-refat/miniconda3/envs/spark/lib/python3.10/site-packages/pyspark/accumulators.py", line 303, in handle
    poll(accum_updates)
  File "/home/ahmed-refat/miniconda3/envs/spark/lib/python3.10/site-packages/pyspark/a

ConnectionRefusedError: [Errno 111] Connection refused